# Synthetic end-to-end validation — L1 abundance recovery (5.02)

The classifier-filter → salmon pipeline on the synthetic chr1 L1-expression benchmark
(model 2: standalone full-length L1 transcripts, per-UID copy count = ground truth).

**Key point — locus resolution is a sequence limit, not a pipeline failure.** Young
full-length L1 are >99% identical, so short reads cannot be assigned to an individual
locus (salmon spreads them across near-identical elements). Recovery therefore improves
with aggregation. This notebook reports R² at three levels — **per-locus → per-subfamily
→ total** — and foregrounds the level at which the model *does* recover abundance.

- Input: `results/synthetic_validation/abundance.csv` (`scripts/sh/synthetic_abundance.py`)
- Figures → `reports/figures/synthetic_validation/`; R² tables → `results/synthetic_validation/`

In [ ]:
library(data.table)
library(ggplot2)

OKABE_ITO <- c("#E69F00", "#56B4E9", "#009E73", "#F0E442",
               "#0072B2", "#D55E00", "#CC79A7", "#000000")
theme_set(theme_bw(base_size = 14))

results_dir <- file.path("..", "results", "synthetic_validation")
figures_dir <- file.path("..", "reports", "figures", "synthetic_validation")
dir.create(figures_dir, recursive = TRUE, showWarnings = FALSE)

save_fig <- function(plot, stem, width, height) {
  for (ext in c("png", "pdf")) {
    suppressMessages(ggsave(file.path(figures_dir, paste0(stem, ".", ext)),
                            plot, width = width, height = height, dpi = 300, bg = "white"))
  }
  invisible(plot)
}

In [ ]:
# --- load abundance table --------------------------------------------------
ab <- fread(file.path(results_dir, "abundance.csv"))
ab[, power := as.integer(power)]
ab[, insertions := 2L^power]
cat(sprintf("rows=%d  cells=%d  elements=%d  subfamilies=%d\n",
            nrow(ab), uniqueN(ab[, .(power, del_prob)]),
            uniqueN(ab$l1_id), uniqueN(ab$subfamily)))
head(ab)

In [ ]:
# --- R^2 at three aggregation levels ---------------------------------------
r2 <- function(x, y) {
  if (length(x) < 3 || sd(x) == 0 || sd(y) == 0) return(NA_real_)
  summary(lm(y ~ x))$r.squared
}

# per-cell aggregates
ab_sub <- ab[, .(sim = sum(simulated), est = sum(albertsalmon_seqlabel)),
             by = .(power, del_prob, subfamily)]           # per-subfamily
ab_tot <- ab[, .(sim = sum(simulated), est = sum(albertsalmon_seqlabel)),
             by = .(power, del_prob)]                        # total (family)

# overall (pooled) R^2 — the aggregation ladder
overall <- data.table(
  level = factor(c("per-locus", "per-subfamily", "total (family)"),
                 levels = c("per-locus", "per-subfamily", "total (family)")),
  r2 = c(r2(ab$simulated, ab$albertsalmon_seqlabel),
         r2(ab_sub$sim, ab_sub$est),
         r2(ab_tot$sim, ab_tot$est)))
fwrite(overall, file.path(results_dir, "r2_by_aggregation.csv"))
print(overall)

# by insertion level: per-locus & per-subfamily vary within a level; total does not
# (its sim = 2^power is constant across del_probs), so it is reported pooled only.
r2_by_level <- rbindlist(list(
  ab[,     .(level = "per-locus",     r2 = r2(simulated, albertsalmon_seqlabel)), by = power],
  ab_sub[, .(level = "per-subfamily", r2 = r2(sim, est)),                          by = power]
))[order(power)]
fwrite(r2_by_level, file.path(results_dir, "r2_by_insertion_rate.csv"))

# total-level R^2 by deletion probability (sim varies across the 9 powers per del_prob)
r2_by_del <- ab_tot[, .(r2 = r2(sim, est)), by = del_prob][order(del_prob)]
fwrite(r2_by_del, file.path(results_dir, "r2_by_del_probability.csv"))
print(r2_by_del)

In [ ]:
# --- Figure: synthetic_albertsalmon_by_level (the aggregation ladder) ------
# R^2 recovered vs simulated rises as near-identical loci are aggregated.
fig_level <- ggplot(overall, aes(level, r2, fill = level)) +
  geom_col(width = 0.62) +
  geom_text(aes(label = sprintf("%.2f", r2)), vjust = -0.4, size = 4.2) +
  scale_fill_manual(values = OKABE_ITO[c(6, 1, 3)], guide = "none") +
  coord_cartesian(ylim = c(0, 1.08)) +
  labs(x = "Aggregation level", y = expression(R^2 ~ "(recovered vs simulated)"))
save_fig(fig_level, "synthetic_albertsalmon_by_level", 6, 4)
fig_level

In [ ]:
# --- Figure: synthetic_albertsalmon_scatter (total / family level) ---------
r2_tot <- overall[level == "total (family)", r2]
fig_scatter <- ggplot(ab_tot, aes(sim, est)) +
  geom_point(size = 2.6, alpha = 0.75, colour = OKABE_ITO[2]) +
  geom_smooth(method = "lm", se = TRUE, colour = OKABE_ITO[6], fill = OKABE_ITO[6]) +
  annotate("text", x = -Inf, y = Inf, hjust = -0.15, vjust = 1.6,
           label = sprintf("R^2 == %.3f", r2_tot), parse = TRUE, size = 5) +
  labs(x = "Simulated total L1 expression (transcript copies)",
       y = "Recovered total (salmon NumReads)")
save_fig(fig_scatter, "synthetic_albertsalmon_scatter", 5, 5)
fig_scatter

In [ ]:
# --- Figure: abundace_by_insertion_rate (legacy 4.08 style) ----------------
# Recovered total abundance (bars) vs the 2^p simulated truth (line), log2 y.
# The recovered NumReads are put on the copy-count scale by one global factor
# (the method recovers relative abundance exactly, R^2 ~ 1), matching the legacy
# "Estimated Abundance vs Insertion Level" figure.
scale <- sum(ab_tot$sim) / sum(ab_tot$est)
rate <- ab_tot[, .(recovered = mean(est) * scale), by = power][order(power)]
rate[, truth := 2^power]
fig_rate <- ggplot(rate, aes(x = factor(power))) +
  geom_col(aes(y = recovered, fill = "AlbertSalmon (recovered)"), width = 0.8) +
  geom_line(aes(y = truth, colour = "Simulated (2^p)", group = 1), linewidth = 1) +
  geom_point(aes(y = truth, colour = "Simulated (2^p)"), size = 2.4) +
  scale_x_discrete(labels = function(p) parse(text = paste0("2^", p))) +
  scale_y_continuous(trans = "log2") +
  scale_fill_manual(values = c("AlbertSalmon (recovered)" = OKABE_ITO[5]), name = NULL) +
  scale_colour_manual(values = c("Simulated (2^p)" = OKABE_ITO[6]), name = NULL) +
  labs(x = "Insertion level (log2)", y = "Estimated abundance (log2)") +
  theme(legend.position = "bottom")
save_fig(fig_rate, "abundace_by_insertion_rate", 8, 5)
fig_rate